In [1]:
!pip install pandas numpy polars scikit-learn lightgbm catboost implicit \
            sentence-transformers librosa transformers pyarrow tqdm \
            scipy matplotlib seaborn faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.1 MB/s eta 0:00:00:00:0100:01
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=933263 sha256=5427dd4be3fbd7bd611f4ddf3304e8a9ce38e5697add589e57cd16a02637c054
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [2]:
from datasets import load_dataset, DatasetDict
import polars as pl
import numpy as np
import pandas as pd
from tqdm import tqdm
import lightgbm as lgb
from catboost import CatBoostRanker
import implicit
from scipy.sparse import csr_matrix
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

In [3]:
def load_yambda_50m_kaggle(
    data_dir: str = "/kaggle/input/datasets/ploshkin/yambda",
    sample_fraction: float = 0.1,
    n_emb_components: int = 64,
    cache: bool = True,
    random_state: int = 42
):
    import polars as pl
    import numpy as np
    from pathlib import Path
    from sklearn.decomposition import PCA
    import gc
    import warnings
    warnings.filterwarnings("ignore")
    
    data_path = Path(data_dir)
    cache_dir = Path("/kaggle/working/yambda_cache")
    
    if cache:
        cache_dir.mkdir(exist_ok=True)
        cache_pos = cache_dir / f"pos_{int(sample_fraction*100)}pct.parquet"
        cache_emb = cache_dir / f"emb_{n_emb_components}d.parquet"
        
        if cache_pos.exists() and cache_emb.exists():
            df_pos = pl.read_parquet(cache_pos)
            df_emb = pl.read_parquet(cache_emb)
            return df_pos, df_emb
    
        
    df_raw = pl.read_parquet("/kaggle/input/datasets/ploshkin/yambda/flat/50m/multi_event.parquet")
    
    df_yambda_pos = df_raw.filter(
        (pl.col("event_type") == "like") | 
        ((pl.col("event_type") == "listen") & (pl.col("played_ratio_pct") >= 70))
    ).select(["uid", "item_id", "timestamp"]).unique()
    
    if sample_fraction < 1.0:
        n_samples = int(len(df_yambda_pos) * sample_fraction)
        df_yambda_pos = df_yambda_pos.sample(n=n_samples, seed=random_state)
    
    df_yambda_pos = df_yambda_pos.sort("timestamp")
    
    del df_raw
    gc.collect()
        
    df_emb_raw = pl.read_parquet(
        "/kaggle/input/datasets/ploshkin/yambda/embeddings.parquet",
        columns=["item_id", "embed"]
    )
    
    target_items = set(df_yambda_pos["item_id"].to_list())
    
    df_emb_filtered = df_emb_raw.filter(pl.col("item_id").is_in(target_items))
    
    embed_list = df_emb_filtered["embed"].to_list()
    embed_matrix = np.vstack(embed_list).astype(np.float32)
    item_ids = df_emb_filtered["item_id"].to_list()
    
    pca = PCA(n_components=n_emb_components, random_state=random_state)
    embed_reduced = pca.fit_transform(embed_matrix).astype(np.float32)
    
    del embed_matrix, embed_list, df_emb_raw, df_emb_filtered
    gc.collect()
    
    df_yambda_emb = pl.DataFrame(
        embed_reduced,
        schema=[f"emb_{i}" for i in range(n_emb_components)],
        schema_overrides={f"emb_{i}": pl.Float32 for i in range(n_emb_components)}
    ).with_columns(pl.Series("item_id", item_ids))
    
    del embed_reduced, item_ids
    gc.collect()
        
    if cache:
        df_yambda_pos.write_parquet(cache_pos, compression="zstd")
        df_yambda_emb.write_parquet(cache_emb, compression="zstd")
    
    print(f"   Взаимодействия: {len(df_yambda_pos):,}")
    print(f"   Пользователей: {df_yambda_pos['uid'].n_unique():,}")
    print(f"   Треков: {df_yambda_pos['item_id'].n_unique():,}")
    print(f"   Эмбеддингов: {len(df_yambda_emb):,} × {n_emb_components}D")
    print(f"   RAM использовано: ~2-4 ГБ")
    
    return df_yambda_pos, df_yambda_emb

In [4]:
df_yambda, df_yambda_emb = load_yambda_50m_kaggle(
    data_dir="/kaggle/input/ploshkin-yambda",
    sample_fraction=0.1,
    n_emb_components=64,
    cache=True
)

🎵 ЗАГРУЗКА YAMBA-50M (Kaggle mode)
   📁 Путь: /kaggle/input/ploshkin-yambda
   📊 Sample: 10%
   🎯 Emb: 512D → 64D

📥 Чтение multi_event.parquet...
   🎲 Сэмплено: 2,885,254 из 2,885,254
   ✅ Позитивных взаимодействий: 2,885,254
   ✅ Уникальных пользователей: 9,799
   ✅ Уникальных треков: 274,373

📥 Чтение embeddings.parquet...
   🎯 Целевых item_id: 274,373
   ✅ Найдено эмбеддингов: 261,886

🔍 Сжатие эмбеддингов (512D → 64D)...
   ✅ Сжато: 64 признаков на 261,886 треков

💾 Сохранение в кэш: /kaggle/working/yambda_cache

✅ YAMBA-50M ЗАГРУЖЕНА (Kaggle)
   📊 Взаимодействия: 2,885,254
   📊 Пользователей: 9,799
   📊 Треков: 274,373
   📊 Эмбеддингов: 261,886 × 64D
   💾 RAM использовано: ~2-4 ГБ


In [8]:
df_pos = df_yambda.sort("timestamp")

print(f"Позитивных взаимодействий готово: {len(df_pos):,}")
print(f"Колонки: {df_pos.columns}")

✅ Позитивных взаимодействий готово: 2,885,254
✅ Колонки: ['uid', 'item_id', 'timestamp']


In [9]:
n = len(df_pos)
train_end, val_end = int(n * 0.7), int(n * 0.85)
df_train = df_pos.slice(0, train_end)
df_val   = df_pos.slice(train_end, val_end - train_end)
df_test  = df_pos.slice(val_end)

In [12]:
df_train = df_train.with_columns(pl.col("timestamp").cast(pl.Int64))

user_feats = df_train.group_by("uid").agg([
    pl.count().alias("user_interactions"),
    pl.col("item_id").n_unique().alias("user_unique_items"),
    (pl.col("timestamp").max() - pl.col("timestamp").min()).alias("user_activity_span"),
    (pl.col("timestamp").max() - pl.col("timestamp").min()).alias("user_last_ts_diff")
])

In [20]:
# FEATURE ENGINEERING

# 4.1. Пользовательские признаки
user_feats = df_train.group_by("uid").agg([
    pl.count().alias("user_interactions"),
    pl.col("item_id").n_unique().alias("user_unique_items"),
    # Конвертируем разность в Int64, чтобы избежать ошибок типов при делении
    (pl.col("timestamp").max() - pl.col("timestamp").min()).cast(pl.Int64).alias("user_activity_span"),
]).with_columns([
    (pl.col("user_interactions") / pl.col("user_unique_items").clip(1)).alias("user_repeat_rate"),
    pl.col("user_activity_span").fill_null(0)
])

# 4.2. Предметные признаки (популярность + уникальность)
item_feats = (df_train
    .group_by("item_id")
    .agg([
        pl.count().alias("item_popularity"),
        pl.col("uid").n_unique().alias("item_unique_users"),
    ])
    .with_columns([
        (pl.col("item_popularity") / pl.col("item_unique_users").clip(1)).alias("item_user_ratio")
    ])
)

# 4.3. Сборка финальных датасетов
def build_dataset1(df_inter, user_df, item_df, emb_df, add_fe=True):
    df = (df_inter
        .join(user_df, on="uid", how="left")
        .join(item_df, on="item_id", how="left")
        .join(emb_df, on="item_id", how="left")
        .fill_null(0)
    )

    # Временные признаки (recency, hour, day)
    df = df.with_columns([
        pl.col("timestamp").cast(pl.Datetime("us")).alias("timestamp_dt"),
        (pl.col("timestamp").max() - pl.col("timestamp")).alias("recency"),
        pl.col("timestamp_dt").dt.hour().alias("hour"),
        pl.col("timestamp").dt.day().alias("day")
    ])

    if not add_fe:
        drop_cols = ["user_interactions", "user_unique_items", "user_activity_span",
                     "user_repeat_rate", "item_popularity", "item_unique_users", "item_user_ratio"]
        df = df.drop([c for c in drop_cols if c in df.columns])

    return df

def build_dataset(df_inter, user_df, item_df, emb_df, add_fe=True):
    df = (df_inter
        .join(user_df, on="uid", how="left")
        .join(item_df, on="item_id", how="left")
        .join(emb_df, on="item_id", how="left")
        .fill_null(0)
    )

    # Создаем временную колонку datetime и считаем recency
    df = df.with_columns([
        pl.col("timestamp").cast(pl.Datetime("us")).alias("timestamp_dt"),
        (df_inter["timestamp"].max() - pl.col("timestamp")).alias("recency"),
    ])

    # Теперь используем timestamp_dt для извлечения часа/дня
    df = df.with_columns([
        pl.col("timestamp_dt").dt.hour().alias("hour"),
        pl.col("timestamp_dt").dt.day().alias("day"),
    ]).drop("timestamp_dt")  # Удаляем временную колонку

    if not add_fe:
        drop_cols = ["user_interactions", "user_unique_items", "user_activity_span",
                     "user_repeat_rate", "item_popularity", "item_unique_users", "item_user_ratio"]
        df = df.drop([c for c in drop_cols if c in df.columns])

    return df

df_train_f = build_dataset(df_train, user_feats, item_feats, df_yambda_emb, add_fe=True)
df_val_f   = build_dataset(df_val, user_feats, item_feats, df_yambda_emb, add_fe=True)

print(f"   Train: {df_train_f.shape}, Val: {df_val_f.shape}")

✅ Feature Engineering завершён
   📊 Train: (2019677, 77), Val: (432788, 77)


In [21]:
def sample_negatives(df_pos, all_items, n_neg=4, seed=42):
    np.random.seed(seed)
    pos_items = df_pos["item_id"].to_numpy()
    pos_uids  = df_pos["uid"].to_numpy()
    pos_ts    = df_pos["timestamp"].to_numpy()

    neg_items = np.random.choice(all_items, size=(len(pos_items), n_neg))

    rows = []
    for i in range(len(pos_items)):
        # негативы
        for ni in neg_items[i]:
            rows.append({"uid": pos_uids[i], "item_id": ni, "timestamp": pos_ts[i], "label": 0})
        # позитив
        rows.append({"uid": pos_uids[i], "item_id": pos_items[i], "timestamp": pos_ts[i], "label": 1})

    return pl.DataFrame(rows)

all_items = df_train["item_id"].unique().to_list()
df_train_ns = sample_negatives(df_train, all_items, n_neg=4)
df_val_ns   = sample_negatives(df_val, all_items, n_neg=4)

In [22]:
def map_at_k(df_preds, k=15):
    df_preds = df_preds.group_by("uid").map_groups(
        lambda x: x.sort("score", descending=True).head(k)
    )

    aps = []
    for group in df_preds.group_by("uid"):
        uid, grp = group
        rel = grp["label"].to_numpy()
        if rel.sum() == 0: continue
        hits = np.cumsum(rel)
        prec = hits / np.arange(1, len(rel) + 1)
        ap = np.sum(prec * rel) / min(k, max(1, rel.sum()))
        aps.append(ap)
    return np.mean(aps) if aps else 0.0

In [25]:
# ЭКСПЕРИМЕНТЫ

features_emb = [c for c in df_train_ns.columns if c.startswith("emb_")]

possible_fe = [
    "total_interactions",
    "unique_items",
    "user_repeat_rate",
    "item_popularity",
    "item_unique_users",
    "item_user_ratio",
    "recency",
    "hour",
    "day"
]

features_fe = [c for c in possible_fe if c in df_train_ns.columns]

def get_Xy(df, features):
    valid_features = [c for c in features if c in df.columns]
    X = df[valid_features].to_numpy()
    y = df["label"].to_numpy()
    return X, y

# LightGBM + emb + FE
X_train, y_train = get_Xy(df_train_ns, features_emb + features_fe)
X_val, y_val     = get_Xy(df_val_ns, features_emb + features_fe)

lgb_train = lgb.Dataset(X_train, y_train, categorical_feature=[])
lgb_val   = lgb.Dataset(X_val, y_val, reference=lgb_train)

params = {"objective": "lambdarank", "metric": "ndcg", "num_leaves": 127,
          "learning_rate": 0.05, "verbose": -1, "seed": 42, "feature_fraction": 0.8}

model_lgb = lgb.train(params, lgb_train, valid_sets=[lgb_val],
                      callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)], num_boost_round=500)

df_val_ns = df_val_ns.with_columns(pl.Series("pred", model_lgb.predict(X_val)))
map1 = map_at_k(df_val_ns[["uid", "item_id", "label", "pred"]])
print(f"LightGBM + emb + FE → MAP@15: {map1:.3f}")

# LightGBM + emb (без FE)
X_train2, y_train2 = get_Xy(df_train_ns, features_emb)
X_val2, y_val2     = get_Xy(df_val_ns, features_emb)

lgb_train2 = lgb.Dataset(X_train2, y_train2)
lgb_val2   = lgb.Dataset(X_val2, y_val2, reference=lgb_train2)
model_lgb2 = lgb.train(params, lgb_train2, valid_sets=[lgb_val2],
                       callbacks=[lgb.early_stopping(30)], num_boost_round=500)

df_val_ns2 = df_val_ns.with_columns(pl.Series("pred", model_lgb2.predict(X_val2)))
map2 = map_at_k(df_val_ns2[["uid", "item_id", "label", "pred"]])
print(f"LightGBM + emb → MAP@15: {map2:.3f}")

# CatBoost + emb + FE
cb_model = CatBoostRanker(iterations=400, learning_rate=0.05, depth=6, loss_function="YetiRank", verbose=False)
cb_model.fit(X_train, y_train, eval_set=(X_val, y_val))

df_val_ns3 = df_val_ns.with_columns(pl.Series("pred", cb_model.predict(X_val)))
map3 = map_at_k(df_val_ns3[["uid", "item_id", "label", "pred"]])
print(f"CatBoost + emb + FE → MAP@15: {map3:.3f}")

# CatBoost + cat_features
cat_candidates = ["item_popularity", "recency"]
cat_feats = [c for c in cat_candidates if c in df_train_ns.columns]

if cat_feats:
    df_train_cat = df_train_ns.with_columns([
        pl.col("item_popularity").cut([10, 50, 100, 500]).alias("pop_bin") if "item_popularity" in df_train_ns.columns else pl.lit(0).alias("pop_bin"),
        pl.col("recency").cut([86400, 604800, 2592000]).alias("recency_bin") if "recency" in df_train_ns.columns else pl.lit(0).alias("recency_bin")
    ]).fill_null(-1)

    drop_cols = ["uid", "item_id", "timestamp", "label", "pop_bin", "recency_bin"]
    X_train4 = df_train_cat.drop([c for c in drop_cols if c in df_train_cat.columns]).to_numpy()
    y_train4 = df_train_cat["label"].to_numpy()
    
    df_val_cat = df_val_ns.with_columns([
        pl.col("item_popularity").cut([10, 50, 100, 500]).alias("pop_bin") if "item_popularity" in df_val_ns.columns else pl.lit(0).alias("pop_bin"),
        pl.col("recency").cut([86400, 604800, 2592000]).alias("recency_bin") if "recency" in df_val_ns.columns else pl.lit(0).alias("recency_bin")
    ]).fill_null(-1)
    
    X_val4 = df_val_cat.drop([c for c in drop_cols if c in df_val_cat.columns]).to_numpy()
    y_val4 = df_val_cat["label"].to_numpy()
    
    cat_indices = [i for i, c in enumerate(df_train_cat.columns) if c in ["pop_bin", "recency_bin"] and c not in ["uid", "item_id", "timestamp", "label"]]
    
    cb_cat = CatBoostRanker(iterations=300, learning_rate=0.05, depth=5, loss_function="YetiRank", verbose=False)
    cb_cat.fit(X_train4, y_train4, eval_set=(X_val4, y_val4), cat_features=cat_indices)
    
    df_val_ns4 = df_val_ns.with_columns(pl.Series("pred", cb_cat.predict(X_val4)))
    map4 = map_at_k(df_val_ns4[["uid", "item_id", "label", "pred"]])
    print(f"CatBoost + cat → MAP@15: {map4:.3f}")
else:
    print("нет колонок для категориальных признаков")
    map4 = 0.0

# ALS + emb
uid2idx = {u:i for i,u in enumerate(df_test["uid"].unique())}
iid2idx = {i:j for j,i in enumerate(df_test["item_id"].unique())}

rows = [uid2idx[u] for u in df_test["uid"]]
cols = [iid2idx[i] for i in df_test["item_id"]]
data = np.ones(len(rows), dtype=np.float32)
mat = csr_matrix((data, (rows, cols)), shape=(len(uid2idx), len(iid2idx)))

als = implicit.als.AlternatingLeastSquares(factors=64, iterations=20, regularization=0.1, use_gpu=False)
als.fit(mat.T)

als_map_scores = []
for uid, uidx in tqdm(uid2idx.items(), desc="ALS Inference"):
    liked = mat[uidx].indices
    recs, _ = als.recommend(uidx, mat[uidx], N=15, filter_already_liked_items=True)
    true_liked = set(df_val.filter(pl.col("uid") == uid)["item_id"].to_list())
    hits = sum(1 for r in recs if r in true_liked)
    if hits == 0: continue
    precisions = np.cumsum([1 if r in true_liked else 0 for r in recs]) / np.arange(1, 16)
    ap = np.sum(precisions * [1 if r in true_liked else 0 for r in recs]) / min(15, len(true_liked))
    als_map_scores.append(ap)

map5 = np.mean(als_map_scores) if als_map_scores else 0.0
print(f"ALS + emb → MAP@15: {map5:.3f}")

🔍 Используемые эмбеддинги: 0
🔍 Используемые FE признаки: []

🔹 Exp 1: LightGBM + emb + FE
X_train shape: (0, 4)
y_train shape: (10098385,)
X_val shape: (0, 4)
y_val shape: (2163940,)


[LightGBM] [Fatal] Check failed: (num_data) > (0) at /__w/1/s/lightgbm-python/src/io/dataset.cpp, line 39 .



LightGBMError: Check failed: (num_data) > (0) at /__w/1/s/lightgbm-python/src/io/dataset.cpp, line 39 .
